### 6. Deployment Pipeline

### `06_deployment_pipeline.ipynb`

In [4]:
## 06_deployment_pipeline_tensorflow.ipynb
import os
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
import pickle
import json
from pathlib import Path
from ultralytics import YOLO
from sklearn.preprocessing import StandardScaler
import time

# Add this after your imports
class FallDetectionEnsemble:
    def __init__(self, tf_model, trad_model, scaler):
        self.tf_model = tf_model
        self.trad_model = trad_model
        self.scaler = scaler
    
    def predict(self, X):
        # Scale data for TensorFlow model
        X_scaled = self.scaler.transform(X)
        
        # Get predictions from both models
        tf_pred_prob = self.tf_model.predict(X_scaled)
        trad_pred_prob = self.trad_model.predict_proba(X)[:, 1]
        
        # Average the probabilities (weighted)
        ensemble_pred_prob = 0.7 * tf_pred_prob.flatten() + 0.3 * trad_pred_prob
        
        # Convert to binary predictions
        return (ensemble_pred_prob >= 0.5).astype(int)
    
    def predict_proba(self, X):
        # Scale data for TensorFlow model
        X_scaled = self.scaler.transform(X)
        
        # Get predictions from both models
        tf_pred_prob = self.tf_model.predict(X_scaled).flatten()
        trad_pred_prob = self.trad_model.predict_proba(X)[:, 1]
        
        # Average the probabilities (weighted)
        ensemble_pred_prob = 0.7 * tf_pred_prob + 0.3 * trad_pred_prob
        
        # Format for sklearn compatibility
        return np.column_stack((1 - ensemble_pred_prob, ensemble_pred_prob))

print("Fall Detection - TensorFlow Deployment Pipeline")

# Set up paths
ROOT_DIR = Path(".")
FEATURES_DIR = ROOT_DIR / "outputs" / "features"
MODELS_DIR = ROOT_DIR / "outputs" / "models"
OUTPUTS_DIR = ROOT_DIR / "outputs" / "deployment"
os.makedirs(OUTPUTS_DIR, exist_ok=True)

# Feature names expected by the model
KEYPOINT_NAMES = [
    "nose", "left_eye", "right_eye", "left_ear", "right_ear", 
    "left_shoulder", "right_shoulder", "left_elbow", "right_elbow", "left_wrist", "right_wrist",
    "left_hip", "right_hip", "left_knee", "right_knee", "left_ankle", "right_ankle"
]

# 1. Load models and configuration
print("\n1. Loading models and configuration...")

# Load model info to determine which model to use
model_info_path = MODELS_DIR / "model_info.json"
if model_info_path.exists():
    with open(model_info_path, 'r') as f:
        model_info = json.load(f)
    print(f"Using model type: {model_info.get('model_type', 'unknown')}")
    model_type = model_info.get('model_type', 'tensorflow')
    model_features = model_info.get('features', [])
else:
    print("Model info not found, defaulting to TensorFlow model")
    model_type = 'tensorflow'
    model_features = []

# Check if analysis findings are available for optimal thresholds
analysis_path = ROOT_DIR / "outputs" / "analysis" / "pattern_analysis_findings.json"
if analysis_path.exists():
    with open(analysis_path, 'r') as f:
        analysis = json.load(f)
    # Get optimal thresholds if available
    thresholds = analysis.get('optimal_thresholds', {'default': 0.5, 'f1_optimal': 0.5, 'f2_optimal': 0.3})
else:
    # Default thresholds
    thresholds = {'default': 0.5, 'f1_optimal': 0.5, 'f2_optimal': 0.3}

print(f"Fall detection thresholds: Default={thresholds['default']:.2f}, F1={thresholds['f1_optimal']:.2f}, F2={thresholds['f2_optimal']:.2f}")

# Load YOLO pose model
try:
    pose_model = YOLO('yolov8n-pose.pt')
    print("YOLO pose model loaded successfully")
except Exception as e:
    print(f"Error loading YOLO pose model: {e}")
    pose_model = None

# Load TensorFlow model if available
tf_model_path = MODELS_DIR / "tensorflow" / "best_model.keras"
if tf_model_path.exists() and (model_type == 'tensorflow' or model_type == 'ensemble'):
    try:
        # Enable unsafe deserialization to load models with lambda layers
        tf.keras.config.enable_unsafe_deserialization()
        
        # Or load with safe_mode=False
        tf_model = tf.keras.models.load_model(tf_model_path, safe_mode=False)
        print(f"TensorFlow model loaded from {tf_model_path}")
    except Exception as e:
        print(f"Error loading TensorFlow model: {e}")
        tf_model = None
else:
    tf_model = None
    print("TensorFlow model not found or not required")

# Load traditional model if needed
trad_model_path = MODELS_DIR / "fall_detection_traditional_model.pkl"
if trad_model_path.exists() and (model_type == 'traditional' or model_type == 'ensemble'):
    try:
        with open(trad_model_path, 'rb') as f:
            trad_model = pickle.load(f)
        print(f"Traditional model loaded from {trad_model_path}")
    except Exception as e:
        print(f"Error loading traditional model: {e}")
        trad_model = None
else:
    trad_model = None
    print("Traditional model not found or not required")

# Load ensemble model if available
ensemble_model_path = MODELS_DIR / "ensemble_model.pkl"
if ensemble_model_path.exists() and model_type == 'ensemble':
    try:
        with open(ensemble_model_path, 'rb') as f:
            ensemble_model = pickle.load(f)
        print(f"Ensemble model loaded from {ensemble_model_path}")
    except Exception as e:
        print(f"Error loading ensemble model: {e}")
        ensemble_model = None
else:
    ensemble_model = None
    print("Ensemble model not found or not required")

# Create or load scaler
scaler_path = MODELS_DIR / "scaler.pkl"
if not scaler_path.exists():
    # Fit with dummy data if we don't have a pre-trained scaler
    # Create a dummy feature set with the same columns
    dummy_features = {}
    for feature in model_features:
        dummy_features[feature] = 0.0
    X_dummy = pd.DataFrame([dummy_features])
    scaler.fit(X_dummy)
    
    # Save the fitted scaler for future use
    with open(scaler_path, 'wb') as f:
        pickle.dump(scaler, f)
    print("Fitted and saved new scaler with default values")

scaler = StandardScaler()

# Create or load scaler
scaler_path = MODELS_DIR / "scaler.pkl"
if scaler_path.exists():
    # Load existing scaler
    try:
        with open(scaler_path, 'rb') as f:
            scaler = pickle.load(f)
        print(f"Loaded scaler from {scaler_path}")
    except Exception as e:
        print(f"Error loading scaler: {e}")
        # Initialize a new scaler if loading fails
        scaler = StandardScaler()
else:
    # Fit with dummy data if we don't have a pre-trained scaler
    scaler = StandardScaler()
    
    # Create a dummy feature set with the same columns
    dummy_features = {}
    for feature in model_features:
        dummy_features[feature] = 0.0
    X_dummy = pd.DataFrame([dummy_features])
    scaler.fit(X_dummy)
    
    # Save the fitted scaler for future use
    with open(scaler_path, 'wb') as f:
        pickle.dump(scaler, f)
    print("Fitted and saved new scaler with default values")

# 2. Helper functions for feature extraction
print("\n2. Setting up feature extraction functions...")

def calculate_joint_angles(keypoints):
    """
    Calculate joint angles from keypoints
    
    Args:
        keypoints: List of keypoints [x, y, confidence]
    
    Returns:
        Dictionary of joint angles in degrees
    """
    # Define joint triplets for angle calculation (joint, center, joint)
    joint_triplets = {
        'right_elbow': (5, 7, 9),    # right shoulder, elbow, wrist
        'left_elbow': (6, 8, 10),    # left shoulder, elbow, wrist
        'right_shoulder': (3, 5, 7),  # right ear, shoulder, elbow
        'left_shoulder': (4, 6, 8),   # left ear, shoulder, elbow
        'right_hip': (5, 11, 13),     # right shoulder, hip, knee
        'left_hip': (6, 12, 14),      # left shoulder, hip, knee
        'right_knee': (11, 13, 15),   # right hip, knee, ankle
        'left_knee': (12, 14, 16),    # left hip, knee, ankle
        'neck': (5, 0, 6)             # right shoulder, nose, left shoulder
    }
    
    angles = {}
    
    # Check if keypoints is a valid list
    if not isinstance(keypoints, list) or len(keypoints) < 17:
        return angles
    
    for joint_name, (p1_idx, p2_idx, p3_idx) in joint_triplets.items():
        # Check if indices are valid
        if max(p1_idx, p2_idx, p3_idx) >= len(keypoints):
            angles[joint_name] = np.nan
            continue
            
        # Check if keypoints have the right format
        try:
            # Skip if any keypoint has low confidence
            if (len(keypoints[p1_idx]) < 3 or len(keypoints[p2_idx]) < 3 or len(keypoints[p3_idx]) < 3 or
                keypoints[p1_idx][2] < 0.5 or keypoints[p2_idx][2] < 0.5 or keypoints[p3_idx][2] < 0.5):
                angles[joint_name] = np.nan
                continue
            
            # Get coordinates
            p1 = keypoints[p1_idx][:2]  # First point
            p2 = keypoints[p2_idx][:2]  # Center point (the joint)
            p3 = keypoints[p3_idx][:2]  # Third point
            
            # Calculate vectors
            v1 = np.array([p1[0] - p2[0], p1[1] - p2[1]])
            v2 = np.array([p3[0] - p2[0], p3[1] - p2[1]])
            
            # Normalize vectors
            v1_norm = np.linalg.norm(v1)
            v2_norm = np.linalg.norm(v2)
            
            if v1_norm == 0 or v2_norm == 0:
                angles[joint_name] = np.nan
                continue
            
            v1 = v1 / v1_norm
            v2 = v2 / v2_norm
            
            # Calculate dot product and angle
            dot_product = np.clip(np.dot(v1, v2), -1.0, 1.0)
            angle = np.arccos(dot_product) * 180 / np.pi
            
            angles[joint_name] = angle
            
        except Exception as e:
            angles[joint_name] = np.nan
    
    return angles

def calculate_bounding_box(keypoints):
    """
    Calculate the bounding box of the pose
    
    Args:
        keypoints: List of keypoints [x, y, confidence]
    
    Returns:
        Dictionary with bounding box properties: x, y, width, height, aspect_ratio
    """
    # Filter valid keypoints
    valid_x = []
    valid_y = []
    
    for kp in keypoints:
        if len(kp) >= 3 and kp[2] >= 0.5:
            valid_x.append(kp[0])
            valid_y.append(kp[1])
    
    if not valid_x or not valid_y:
        return {
            'x': np.nan, 'y': np.nan, 
            'width': np.nan, 'height': np.nan, 
            'area': np.nan, 'aspect_ratio': np.nan
        }
    
    # Calculate bounding box
    min_x = min(valid_x)
    max_x = max(valid_x)
    min_y = min(valid_y)
    max_y = max(valid_y)
    
    width = max_x - min_x
    height = max_y - min_y
    area = width * height
    aspect_ratio = width / height if height > 0 else np.nan
    
    return {
        'x': min_x,
        'y': min_y,
        'width': width,
        'height': height,
        'area': area,
        'aspect_ratio': aspect_ratio
    }

def calculate_velocity(positions, fps):
    """
    Calculate velocity from position data
    
    Args:
        positions: List of position values over time
        fps: Frames per second
    
    Returns:
        List of velocities
    """
    if len(positions) < 2:
        return []
    
    # Convert to numpy array for easier processing
    positions = np.array(positions)
    
    # Calculate velocity (change in position per second)
    velocities = np.diff(positions) * fps
    
    # Add 0 as first velocity for consistent length
    return np.concatenate(([0], velocities)).tolist()

def extract_features_from_pose_sequence(pose_sequence, fps=30):
    """
    Extract features from a sequence of poses for fall detection
    
    Args:
        pose_sequence: List of keypoints over time
        fps: Frames per second
    
    Returns:
        Dictionary of features for fall detection
    """
    # Skip if sequence is too short
    if len(pose_sequence) < 3:
        return {}
        
    # Initialize features
    features = {}
    
    # Calculate joint angles for most recent frame
    joint_angles = calculate_joint_angles(pose_sequence[-1])
    
    # Add joint angles to features
    for joint in ['right_elbow', 'left_elbow', 'right_knee', 'left_knee', 'right_hip', 'left_hip']:
        features[f"{joint}_mean"] = joint_angles.get(joint, 0)
    
    # Calculate bounding box for most recent frame
    bbox = calculate_bounding_box(pose_sequence[-1])
    if not np.isnan(bbox['aspect_ratio']):
        features['aspect_ratio'] = bbox['aspect_ratio']
    else:
        features['aspect_ratio'] = 1.0
    
    # Track vertical positions over time
    upper_body_y = []
    center_y = []
    
    for frame_keypoints in pose_sequence:
        # Upper body markers (nose, shoulders)
        upper_markers = [0, 5, 6]  # Indices for nose, right shoulder, left shoulder
        
        # Calculate average y-position of upper body
        valid_y = []
        for idx in upper_markers:
            if idx < len(frame_keypoints) and len(frame_keypoints[idx]) >= 3 and frame_keypoints[idx][2] >= 0.5:
                valid_y.append(frame_keypoints[idx][1])
        
        if valid_y:
            upper_body_y.append(np.mean(valid_y))
        else:
            upper_body_y.append(np.nan)
        
        # Calculate overall center y-position
        valid_center_y = []
        for kp in frame_keypoints:
            if len(kp) >= 3 and kp[2] >= 0.5:
                valid_center_y.append(kp[1])
        
        if valid_center_y:
            center_y.append(np.mean(valid_center_y))
        else:
            center_y.append(np.nan)
    
    # Calculate velocity
    upper_body_velocity_y = calculate_velocity(upper_body_y, fps)
    
    # Add motion features
    if upper_body_velocity_y:
        features['max_upper_body_velocity_y'] = max(abs(v) for v in upper_body_velocity_y if not np.isnan(v))
    else:
        features['max_upper_body_velocity_y'] = 0
    
    # Add height change features
    if len(upper_body_y) > 1 and not np.isnan(upper_body_y[0]) and not np.isnan(upper_body_y[-1]):
        features['upper_body_vertical_displacement'] = upper_body_y[-1] - upper_body_y[0]
    else:
        features['upper_body_vertical_displacement'] = 0
    
    # Track height and aspect ratio over time
    height_values = []
    aspect_ratios = []
    
    for frame_keypoints in pose_sequence:
        bbox = calculate_bounding_box(frame_keypoints)
        height_values.append(bbox['height'])
        aspect_ratios.append(bbox['aspect_ratio'])
    
    # Calculate height reduction percentage
    if len(height_values) > 1:
        valid_indices = [i for i, h in enumerate(height_values) if not np.isnan(h)]
        if valid_indices:
            first_valid_height = height_values[valid_indices[0]]
            min_height = min([h for h in height_values if not np.isnan(h)])
            if first_valid_height > 0:
                features['height_reduction_pct'] = (first_valid_height - min_height) / first_valid_height * 100
            else:
                features['height_reduction_pct'] = 0
        else:
            features['height_reduction_pct'] = 0
    else:
        features['height_reduction_pct'] = 0
    
    # Calculate aspect ratio increase percentage
    if len(aspect_ratios) > 1:
        valid_indices = [i for i, ar in enumerate(aspect_ratios) if not np.isnan(ar)]
        if valid_indices:
            first_valid_ar = aspect_ratios[valid_indices[0]]
            max_ar = max([ar for ar in aspect_ratios if not np.isnan(ar)])
            if first_valid_ar > 0:
                features['aspect_ratio_increase_pct'] = (max_ar - first_valid_ar) / first_valid_ar * 100
            else:
                features['aspect_ratio_increase_pct'] = 0
        else:
            features['aspect_ratio_increase_pct'] = 0
    else:
        features['aspect_ratio_increase_pct'] = 0
        
    # Calculate height velocity
    height_velocity = calculate_velocity(height_values, fps)
    if height_velocity:
        features['max_height_collapse_velocity'] = min([v for v in height_velocity if not np.isnan(v)], default=0)
    else:
        features['max_height_collapse_velocity'] = 0
    
    # Add impact features
    features['has_impact'] = 0  # Default to no impact
    
    return features

# 3. Define prediction functions
print("\n3. Setting up prediction functions...")

def predict_with_model(features, model_type='tensorflow'):
    """
    Make a fall detection prediction using the loaded model
    
    Args:
        features: Dictionary of extracted features
        model_type: Type of model to use (tensorflow, traditional, ensemble)
    
    Returns:
        prediction, probability
    """
    # Filter features to include only those expected by the model
    filtered_features = {}
    for feature in model_features:
        if feature in features:
            filtered_features[feature] = features[feature]
        else:
            # Use default value for missing features
            filtered_features[feature] = 0.0
    
    # Convert to DataFrame for prediction
    X = pd.DataFrame([filtered_features])
    
    # Scale features if needed
    X_scaled = scaler.transform(X)
    
    # Make prediction based on model type
    if model_type == 'tensorflow' and tf_model is not None:
        # TensorFlow prediction
        probability = tf_model.predict(X_scaled)[0][0]
        prediction = int(probability >= thresholds['default'])
        return prediction, probability
    
    elif model_type == 'traditional' and trad_model is not None:
        # Traditional ML prediction
        prediction = trad_model.predict(X)[0]
        probability = trad_model.predict_proba(X)[0][1]
        return prediction, probability
    
    elif model_type == 'ensemble' and ensemble_model is not None:
        # Ensemble prediction
        # Get predictions from both models
        X_scaled = scaler.transform(X)
        tf_pred_prob = tf_model.predict(X_scaled).flatten()[0]
        trad_pred_prob = trad_model.predict_proba(X)[0][1]
        
        # Average the probabilities (weighted)
        probability = 0.7 * tf_pred_prob + 0.3 * trad_pred_prob
        prediction = int(probability >= thresholds['default'])
        return prediction, probability
    
    else:
        # Fallback to a simple heuristic if no models are available
        print("No models available, using simple heuristic")
        vertical_velocity = features.get('max_upper_body_velocity_y', 0)
        height_reduction = features.get('height_reduction_pct', 0)
        
        # Simple heuristic: high velocity + significant height reduction might be a fall
        if abs(vertical_velocity) > 50 and height_reduction > 30:
            return 1, 0.9
        elif abs(vertical_velocity) > 30 and height_reduction > 20:
            return 1, 0.7
        else:
            return 0, 0.3

# 4. Define the real-time fall detection pipeline
print("\n4. Defining real-time fall detection pipeline...")

def real_time_fall_detection(video_source=0, output_path=None, threshold_type='default'):
    """
    Real-time fall detection using the loaded model
    
    Args:
        video_source: Camera index or path to video file
        output_path: Path to save the output video
        threshold_type: Type of threshold to use ('default', 'f1_optimal', 'f2_optimal')
    
    Returns:
        None
    """
    # Check if pose model is available
    if pose_model is None:
        print("YOLO pose model not available. Cannot detect poses.")
        return
    
    # Select threshold
    threshold = thresholds.get(threshold_type, 0.5)
    print(f"Using {threshold_type} threshold: {threshold:.2f}")
    
    # Open video capture
    cap = cv2.VideoCapture(video_source)
    if not cap.isOpened():
        print(f"Error: Could not open video source {video_source}")
        return
    
    # Get video properties
    fps = cap.get(cv2.CAP_PROP_FPS)
    frame_width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    frame_height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    
    # Set up video writer if output path is specified
    if output_path:
        fourcc = cv2.VideoWriter_fourcc(*'mp4v')
        out = cv2.VideoWriter(output_path, fourcc, fps, (frame_width, frame_height))
    
    # Initialize variables for tracking
    pose_sequence = []
    fall_probabilities = []
    fall_detected = False
    fall_duration = 0
    fall_start_time = None
    
    # Font settings for display
    font = cv2.FONT_HERSHEY_SIMPLEX
    font_scale = 0.7
    thickness = 2
    
    # Initialize performance tracking
    frame_times = []
    
    # Main loop
    while True:
        start_time = time.time()
        
        # Read frame
        ret, frame = cap.read()
        if not ret:
            break
        
        # Run YOLO pose detection
        results = pose_model(frame)
        
        # Create copy for annotation
        display_frame = frame.copy()
        
        # Process detection results
        keypoints = None
        if results and len(results) > 0:
            # Draw pose keypoints
            annotated_frame = results[0].plot()
            display_frame = annotated_frame
            
            # Extract keypoints
            if hasattr(results[0], 'keypoints') and results[0].keypoints is not None:
                keypoints_data = results[0].keypoints.data
                
                if len(keypoints_data) > 0:
                    # Convert keypoints to the expected format
                    keypoints = []
                    for kp_idx in range(len(keypoints_data[0])):
                        kp = keypoints_data[0][kp_idx].cpu().numpy()
                        # Each keypoint: [x, y, confidence]
                        keypoints.append([float(kp[0]), float(kp[1]), float(kp[2])])
        
        # If keypoints detected, update sequence and check for falls
        if keypoints:
            # Add to pose sequence
            pose_sequence.append(keypoints)
            
            # Keep only the last 30 frames (about 1 second at 30 fps)
            if len(pose_sequence) > 30:
                pose_sequence.pop(0)
            
            # If we have enough frames, extract features and predict
            if len(pose_sequence) >= 5:
                # Extract features
                features = extract_features_from_pose_sequence(pose_sequence, fps)
                
                # Make prediction
                prediction, probability = predict_with_model(features, model_type)
                
                # Add to probability history for smoothing
                fall_probabilities.append(probability)
                if len(fall_probabilities) > 10:  # Keep last 10 probabilities
                    fall_probabilities.pop(0)
                
                # Smooth the probability by averaging recent values
                smoothed_probability = sum(fall_probabilities) / len(fall_probabilities)
                
                # Check if fall is detected based on smoothed probability
                current_detection = smoothed_probability >= threshold
                
                # Handle fall detection state
                if current_detection:
                    if not fall_detected:
                        # New fall detected
                        fall_detected = True
                        fall_start_time = time.time()
                    else:
                        # Continuing fall
                        fall_duration = time.time() - fall_start_time
                else:
                    # No fall detected
                    if fall_detected and fall_duration > 0.5:  # Only reset if fall lasted more than 0.5 seconds
                        print(f"Fall event ended. Duration: {fall_duration:.1f} seconds")
                    fall_detected = False
                    fall_duration = 0
                    fall_start_time = None
                
                # Display probability
                status = f"Fall probability: {smoothed_probability:.2f}"
                color = (0, 0, 255) if current_detection else (0, 255, 0)
                cv2.putText(display_frame, status, (10, 30), font, font_scale, color, thickness)
                
                # Add alert for fall detection
                if fall_detected:
                    cv2.putText(display_frame, "FALL DETECTED!", (frame_width//4, frame_height//2),
                                font, 2, (0, 0, 255), 3)
                    cv2.rectangle(display_frame, (0, 0), (frame_width, frame_height), (0, 0, 255), 10)
                    
                    # Also display duration if fall is ongoing
                    if fall_duration > 0:
                        duration_text = f"Duration: {fall_duration:.1f}s"
                        cv2.putText(display_frame, duration_text, (frame_width//4, frame_height//2 + 50),
                                   font, 1, (0, 0, 255), 2)
                
                # Display feature values (top 3 features)
                if model_features:
                    for i, feature in enumerate(model_features[:3]):
                        if feature in features:
                            feature_text = f"{feature}: {features[feature]:.2f}"
                            cv2.putText(display_frame, feature_text, (10, 60 + i*30), font, 0.5, (255, 255, 255), 1)
        
        # Calculate frame processing time
        end_time = time.time()
        frame_time = end_time - start_time
        frame_times.append(frame_time)
        
        # Display FPS
        avg_frame_time = sum(frame_times[-30:]) / len(frame_times[-30:])
        fps_text = f"FPS: {1.0 / avg_frame_time:.1f}"
        cv2.putText(display_frame, fps_text, (frame_width - 120, 30), font, font_scale, (0, 255, 255), thickness)
        
        # Display the frame
        cv2.imshow('Fall Detection', display_frame)
        
        # Write to output if specified
        if output_path and 'out' in locals():
            out.write(display_frame)
        
        # Exit on 'q' key press
        if cv2.waitKey(1) & 0xFF == ord('q'):
            break
    
    # Clean up
    cap.release()
    if output_path and 'out' in locals():
        out.release()
    cv2.destroyAllWindows()
    
    # Report performance
    if frame_times:
        avg_fps = 1.0 / (sum(frame_times) / len(frame_times))
        print(f"Average FPS: {avg_fps:.1f}")
    
    return

# 5. Define batch processing for videos
print("\n5. Defining batch processing for videos...")

def batch_process_videos(video_dir, output_dir=None, threshold_type='default'):
    """
    Process a batch of videos for fall detection
    
    Args:
        video_dir: Directory containing videos to process
        output_dir: Directory to save the output videos
        threshold_type: Type of threshold to use ('default', 'f1_optimal', 'f2_optimal')
    
    Returns:
        results: Dictionary with results for each video
    """
    # Create output directory if not exists
    if output_dir:
        os.makedirs(output_dir, exist_ok=True)
    
    # Find videos in directory
    video_extensions = ['.mp4', '.avi', '.mov']
    videos = []
    for ext in video_extensions:
        videos.extend(list(Path(video_dir).glob(f"*{ext}")))
    
    print(f"Found {len(videos)} videos in {video_dir}")
    
    # Results dictionary
    results = {}
    
    # Process each video
    for video_path in videos:
        video_name = video_path.stem
        print(f"Processing {video_name}...")
        
        # Define output path if needed
        output_path = None
        if output_dir:
            output_path = Path(output_dir) / f"{video_name}_detection.mp4"
        
        # Process video
        try:
            video_results = process_video(str(video_path), output_path, threshold_type)
            results[video_name] = video_results
        except Exception as e:
            print(f"Error processing {video_name}: {e}")
            results[video_name] = {'error': str(e)}
    
    # Save results to JSON
    if output_dir:
        results_path = Path(output_dir) / "batch_results.json"
        with open(results_path, 'w') as f:
            json.dump(results, f, indent=2)
    
    return results

def process_video(video_path, output_path=None, threshold_type='default'):
    """
    Process a single video file for fall detection
    
    Args:
        video_path: Path to the video file
        output_path: Path to save the output video
        threshold_type: Type of threshold to use ('default', 'f1_optimal', 'f2_optimal')
    
    Returns:
        results: Dictionary with detection results
    """
    
    # Add font definitions here
    font = cv2.FONT_HERSHEY_SIMPLEX
    font_scale = 0.7
    thickness = 2
    
    # Check if pose model is available
    if pose_model is None:
        return {'error': "YOLO pose model not available"}
    
    # Select threshold
    threshold = thresholds.get(threshold_type, 0.5)
    
    # Open video
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        return {'error': f"Could not open video file {video_path}"}
    
    # Get video properties
    fps = cap.get(cv2.CAP_PROP_FPS)
    frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    frame_width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    frame_height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    
    # Set up video writer if output path is specified
    if output_path:
        fourcc = cv2.VideoWriter_fourcc(*'mp4v')
        out = cv2.VideoWriter(str(output_path), fourcc, fps, (frame_width, frame_height))
    
    # Initialize variables
    pose_sequence = []
    fall_frames = []
    fall_probabilities = []
    max_probability = 0
    current_fall_event = None
    fall_events = []
    
    # Process frames
    frame_idx = 0
    while True:
        ret, frame = cap.read()
        if not ret:
            break
        
        # Run YOLO pose detection
        results = pose_model(frame)
        
        # Create copy for annotation
        display_frame = frame.copy()
        
        # Extract keypoints
        keypoints = None
        if results and len(results) > 0:
            # Draw pose keypoints
            annotated_frame = results[0].plot()
            display_frame = annotated_frame
            
            # Extract keypoints
            if hasattr(results[0], 'keypoints') and results[0].keypoints is not None:
                keypoints_data = results[0].keypoints.data
                
                if len(keypoints_data) > 0:
                    # Convert keypoints to the expected format
                    keypoints = []
                    for kp_idx in range(len(keypoints_data[0])):
                        kp = keypoints_data[0][kp_idx].cpu().numpy()
                        keypoints.append([float(kp[0]), float(kp[1]), float(kp[2])])
        
        # If keypoints detected, update sequence and check for falls
        probability = 0
        detection = False
        
        if keypoints:
            # Add to pose sequence
            pose_sequence.append(keypoints)
            
            # Keep only the last 30 frames
            if len(pose_sequence) > 30:
                pose_sequence.pop(0)
            
            # If we have enough frames, extract features and predict
            if len(pose_sequence) >= 5:
                # Extract features
                features = extract_features_from_pose_sequence(pose_sequence, fps)
                
                # Make prediction
                prediction, probability = predict_with_model(features, model_type)
                
                # Add to probability history for smoothing
                fall_probabilities.append(probability)
                if len(fall_probabilities) > 10:  # Keep last 10 probabilities
                    fall_probabilities.pop(0)
                
                # Smooth the probability by averaging recent values
                smoothed_probability = sum(fall_probabilities) / len(fall_probabilities)
                
                # Check if this is the highest probability so far
                max_probability = max(max_probability, smoothed_probability)
                
                # Check if fall is detected based on smoothed probability
                detection = smoothed_probability >= threshold
                
                # Track fall events
                if detection:
                    # Add frame to fall frames
                    fall_frames.append(frame_idx)
                    
                    # Check if this is part of an ongoing fall event
                    if current_fall_event is None:
                        # Start a new fall event
                        current_fall_event = {
                            'start_frame': frame_idx,
                            'end_frame': frame_idx,
                            'max_probability': smoothed_probability,
                            'duration_sec': 1 / fps
                        }
                    else:
                        # Update current fall event
                        current_fall_event['end_frame'] = frame_idx
                        current_fall_event['max_probability'] = max(current_fall_event['max_probability'], smoothed_probability)
                        current_fall_event['duration_sec'] = (frame_idx - current_fall_event['start_frame']) / fps
                else:
                    # No detection, check if we need to end an event
                    if current_fall_event is not None:
                        # Add the completed event to the list and reset
                        fall_events.append(current_fall_event)
                        current_fall_event = None
                
                # Add annotations
                status = f"Fall probability: {smoothed_probability:.2f}"
                color = (0, 0, 255) if detection else (0, 255, 0)
                cv2.putText(display_frame, status, (10, 30), font, font_scale, color, thickness)
                
                # Add alert for fall detection
                if detection:
                    cv2.putText(display_frame, "FALL DETECTED!", (frame_width//4, frame_height//2),
                                font, 2, (0, 0, 255), 3)
                    cv2.rectangle(display_frame, (0, 0), (frame_width, frame_height), (0, 0, 255), 10)
        
        # Add frame counter
        frame_text = f"Frame: {frame_idx}/{frame_count}"
        cv2.putText(display_frame, frame_text, (10, frame_height - 10), font, 0.5, (255, 255, 255), 1)
        
        # Write to output if specified
        if output_path and 'out' in locals():
            out.write(display_frame)
        
        frame_idx += 1
    
    # Clean up
    cap.release()
    if output_path and 'out' in locals():
        out.release()
    
    # Finalize the last fall event if exists
    if current_fall_event is not None:
        fall_events.append(current_fall_event)
    
    # Merge fall events that are close together (< 1 second apart)
    merged_events = []
    if fall_events:
        current_merged = fall_events[0]
        
        for i in range(1, len(fall_events)):
            current_event = fall_events[i]
            frame_gap = current_event['start_frame'] - current_merged['end_frame']
            time_gap = frame_gap / fps
            
            if time_gap < 1.0:  # Merge if less than 1 second apart
                # Extend the current merged event
                current_merged['end_frame'] = current_event['end_frame']
                current_merged['max_probability'] = max(current_merged['max_probability'], current_event['max_probability'])
                current_merged['duration_sec'] = (current_merged['end_frame'] - current_merged['start_frame']) / fps
            else:
                # Add the current merged event and start a new one
                merged_events.append(current_merged)
                current_merged = current_event
        
        # Add the last merged event
        merged_events.append(current_merged)
    
    # Calculate results
    has_fall = len(merged_events) > 0
    
    # Create results dictionary
    results = {
        'video_path': video_path,
        'has_fall': has_fall,
        'max_probability': float(max_probability),
        'fall_events_count': len(merged_events),
        'fall_events': merged_events,
        'threshold_used': threshold,
        'threshold_type': threshold_type,
        'model_type': model_type
    }
    
    print(f"Processed {video_path}: {len(merged_events)} fall events detected")
    
    return results

# 6. Main Execution
if __name__ == "__main__":
    print("\n6. Fall Detection Demo")
    print("Choose an option:")
    print("1. Real-time Fall Detection (Webcam)")
    print("2. Process a Video File")
    print("3. Batch Process Videos")
    
    choice = input("Enter choice (1-3): ")
    
    if choice == '1':
        # Real-time detection with webcam
        threshold_choice = input("Select threshold (1-Default, 2-F1 Optimal, 3-F2 Optimal for Higher Recall): ")
        if threshold_choice == '2':
            threshold_type = 'f1_optimal'
        elif threshold_choice == '3':
            threshold_type = 'f2_optimal'
        else:
            threshold_type = 'default'
        
        # Output file
        save_output = input("Save output video? (y/n): ").lower().startswith('y')
        output_path = None
        if save_output:
            output_path = str(OUTPUTS_DIR / "webcam_detection.mp4")
        
        print("Starting real-time fall detection (press 'q' to quit)...")
        real_time_fall_detection(0, output_path, threshold_type)
        
    elif choice == '2':
        # Process a video file
        video_path = input("Enter path to video file: ")
        if not os.path.exists(video_path):
            print(f"Error: File {video_path} not found")
        else:
            threshold_choice = input("Select threshold (1-Default, 2-F1 Optimal, 3-F2 Optimal for Higher Recall): ")
            if threshold_choice == '2':
                threshold_type = 'f1_optimal'
            elif threshold_choice == '3':
                threshold_type = 'f2_optimal'
            else:
                threshold_type = 'default'
            
            output_path = str(OUTPUTS_DIR / f"{Path(video_path).stem}_detection.mp4")
            print(f"Processing {video_path}...")
            results = process_video(video_path, output_path, threshold_type)
            
            # Print results
            print("\nDetection Results:")
            if 'error' in results:
                print(f"Error: {results['error']}")
            else:
                print(f"Fall detected: {results['has_fall']}")
                print(f"Maximum fall probability: {results['max_probability']:.2f}")
                print(f"Number of fall events: {results['fall_events_count']}")
                
                # Print details for each fall event
                if results['fall_events_count'] > 0:
                    print("\nFall Events:")
                    for i, event in enumerate(results['fall_events']):
                        print(f"Event {i+1}:")
                        print(f"  Frames: {event['start_frame']} - {event['end_frame']}")
                        print(f"  Duration: {event['duration_sec']:.2f} seconds")
                        print(f"  Max Probability: {event['max_probability']:.2f}")
            
            print(f"\nOutput video saved to {output_path}")
            
    elif choice == '3':
        # Batch process videos
        video_dir = input("Enter directory containing videos: ")
        if not os.path.exists(video_dir) or not os.path.isdir(video_dir):
            print(f"Error: Directory {video_dir} not found")
        else:
            threshold_choice = input("Select threshold (1-Default, 2-F1 Optimal, 3-F2 Optimal for Higher Recall): ")
            if threshold_choice == '2':
                threshold_type = 'f1_optimal'
            elif threshold_choice == '3':
                threshold_type = 'f2_optimal'
            else:
                threshold_type = 'default'
            
            output_dir = str(OUTPUTS_DIR / "batch_results")
            print(f"Processing videos in {video_dir}...")
            results = batch_process_videos(video_dir, output_dir, threshold_type)
            
            # Print summary
            total_videos = len(results)
            videos_with_falls = sum(1 for v in results.values() if isinstance(v, dict) and v.get('has_fall', False))
            
            print("\nBatch Processing Summary:")
            print(f"Total videos processed: {total_videos}")
            print(f"Videos with falls detected: {videos_with_falls}")
            print(f"Detection rate: {videos_with_falls/total_videos*100:.1f}%")
            print(f"Results saved to {output_dir}")
    
    else:
        print("Invalid choice")

Fall Detection - TensorFlow Deployment Pipeline

1. Loading models and configuration...
Using model type: ensemble
Fall detection thresholds: Default=0.50, F1=0.60, F2=0.60
YOLO pose model loaded successfully
TensorFlow model loaded from outputs\models\tensorflow\best_model.keras
Traditional model loaded from outputs\models\fall_detection_traditional_model.pkl
Ensemble model loaded from outputs\models\ensemble_model.pkl
Loaded scaler from outputs\models\scaler.pkl

2. Setting up feature extraction functions...

3. Setting up prediction functions...

4. Defining real-time fall detection pipeline...

5. Defining batch processing for videos...

6. Fall Detection Demo
Choose an option:
1. Real-time Fall Detection (Webcam)
2. Process a Video File
3. Batch Process Videos


Enter choice (1-3):  3
Enter directory containing videos:  test_videos
Select threshold (1-Default, 2-F1 Optimal, 3-F2 Optimal for Higher Recall):  3


Processing videos in test_videos...
Found 11 videos in test_videos
Processing barbell biceps curl_1...

0: 384x640 1 person, 60.1ms
Speed: 3.1ms preprocess, 60.1ms inference, 1.1ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 1 person, 56.0ms
Speed: 1.7ms preprocess, 56.0ms inference, 1.0ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 1 person, 53.1ms
Speed: 1.6ms preprocess, 53.1ms inference, 1.1ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 1 person, 57.9ms
Speed: 1.8ms preprocess, 57.9ms inference, 0.8ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 1 person, 54.4ms
Speed: 1.6ms preprocess, 54.4ms inference, 0.5ms postprocess per image at shape (1, 3, 384, 640)
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 79ms/step

0: 384x640 1 person, 58.7ms
Speed: 2.9ms preprocess, 58.7ms inference, 0.9ms postprocess per image at shape (1, 3, 384, 640)
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 26ms/step

0: 384x640 1 person, 52.7ms
Speed: 1.3ms preprocess, 52.7ms infe